In [1]:
from src.cluster_metrics import load_and_scale, sweep_k, plot_metric_curves, plot_silhouette_per_k, plot_pca_grid, print_summary, export_player_archetypes

In [2]:
players_df, X_scaled, scaler = load_and_scale()

Loading player_stats.csv...
  444 players with 15+ games


In [3]:
summary_df, all_labels = sweep_k(X_scaled)

  Fitting K=2... sil=0.315  DB=1.315  CH=194  weakest_cluster_sil=0.219
  Fitting K=3... sil=0.290  DB=1.157  CH=195  weakest_cluster_sil=0.248
  Fitting K=4... sil=0.233  DB=1.321  CH=167  weakest_cluster_sil=0.171
  Fitting K=5... sil=0.210  DB=1.362  CH=151  weakest_cluster_sil=0.189
  Fitting K=6... sil=0.218  DB=1.316  CH=143  weakest_cluster_sil=0.118
  Fitting K=7... sil=0.195  DB=1.293  CH=133  weakest_cluster_sil=0.101
  Fitting K=8... sil=0.212  DB=1.252  CH=128  weakest_cluster_sil=0.128
  Fitting K=9... sil=0.188  DB=1.294  CH=119  weakest_cluster_sil=0.056
  Fitting K=10... sil=0.193  DB=1.286  CH=116  weakest_cluster_sil=0.100
  Fitting K=11... sil=0.182  DB=1.321  CH=111  weakest_cluster_sil=0.099
  Fitting K=12... sil=0.188  DB=1.304  CH=108  weakest_cluster_sil=0.099


In [4]:
plot_metric_curves(summary_df)
plot_silhouette_per_k(X_scaled, all_labels)
plot_pca_grid(X_scaled, all_labels, players_df)


  Saved cluster_evaluation_metrics.png
  Saved cluster_silhouette_bars.png
  Saved cluster_pca_grid.png


In [5]:
print_summary(summary_df)



CLUSTER EVALUATION SUMMARY
 k     inertia  silhouette  davies_bouldin  calinski_harabasz  weakest_cluster_sil  smallest_cluster_size
 2 2159.344399    0.314561        1.315113         194.181982             0.219270                    133
 3 1649.447403    0.290226        1.157067         194.980966             0.248457                     99
 4 1453.030748    0.232844        1.320601         167.050026             0.170973                     52
 5 1308.485914    0.209606        1.361541         150.935267             0.188865                     47
 6 1180.854806    0.218270        1.315605         142.962469             0.117856                     27
 7 1097.620823    0.195286        1.293359         133.399999             0.100646                     25
 8 1016.875960    0.211984        1.251801         128.085587             0.128116                     15
 9  973.454097    0.188011        1.294002         119.231029             0.056021                     22
10  912.388689    

In [6]:
# ── Once you pick a K, call this to export final assignments ──────────────
# Change the number below to whichever K you decide on after reviewing plots
CHOSEN_K = int(summary_df.loc[summary_df['silhouette'].idxmax(), 'k'])
print(f"\nAuto-exporting best K by silhouette = {CHOSEN_K}...")
export_player_archetypes(players_df, X_scaled, CHOSEN_K, scaler)

print("\nDone. Files saved:")
print("  cluster_evaluation_metrics.png  — 4 metric curves")
print("  cluster_silhouette_bars.png     — per-cluster silhouette for every K")
print("  cluster_pca_grid.png            — PCA view for every K")
print("  cluster_evaluation_summary.csv  — numeric table")
print(f"  player_archetypes_k{CHOSEN_K}.csv       — player assignments for best K")


Auto-exporting best K by silhouette = 2...

── Centroids for K=2 ─────────────────────────────────────────────
         USG_PCT  AST_PCT  OREB_PCT  DREB_PCT  TS_PCT    PIE  REB_PCT
cluster                                                              
0          0.181    0.158     0.032     0.104   0.551  0.083    0.068
1          0.181    0.130     0.085     0.182   0.608  0.115    0.134

── Top 5 players per cluster (K=2) ──────────────────────────────
  C0: Shai Gilgeous-Alexander, Cade Cunningham, Donovan Mitchell, Tyrese Maxey, Anthony Edwards
  C1: Nikola Jokić, Giannis Antetokounmpo, Victor Wembanyama, Kawhi Leonard, Luka Dončić

  Saved player_archetypes_k2.csv

Done. Files saved:
  cluster_evaluation_metrics.png  — 4 metric curves
  cluster_silhouette_bars.png     — per-cluster silhouette for every K
  cluster_pca_grid.png            — PCA view for every K
  cluster_evaluation_summary.csv  — numeric table
  player_archetypes_k2.csv       — player assignments for best K


### New approach: using HDBSCAN and UMAP

In [7]:
from src.umap import load_players, run_sweep, plot_best

In [8]:
df = load_players()
results_df = run_sweep(df)
valid = results_df[results_df['status'] == 'OK'].sort_values('score', ascending=False)

Loaded 444 players with 15+ games

  [  1/108] feat=base | nn=10 md=0.05 | mcs=20 ms=3  →  k=9  noise=35  sil=0.5012  score=0.5012
  [  2/108] feat=base | nn=10 md=0.05 | mcs=20 ms=5  →  k=8  noise=34  sil=0.5109  score=0.5109
  [  3/108] feat=base | nn=10 md=0.05 | mcs=25 ms=5  →  k=8  noise=34  sil=0.5109  score=0.5109
  [  4/108] feat=base | nn=10 md=0.05 | mcs=30 ms=5  →  k=6  noise=17  sil=0.5093  score=0.5093
  [  5/108] feat=base | nn=10 md=0.05 | mcs=30 ms=8  →  k=4  noise=17  sil=0.4239  score=0.4239
  [  6/108] feat=base | nn=10 md=0.05 | mcs=35 ms=5  →  k=6  noise=17  sil=0.5093  score=0.5093
  [  7/108] feat=base | nn=10 md=0.1 | mcs=20 ms=3  →  k=9  noise=42  sil=0.4801  score=0.4801
  [  8/108] feat=base | nn=10 md=0.1 | mcs=20 ms=5  →  k=7  noise=37  sil=0.4785  score=0.4785
  [  9/108] feat=base | nn=10 md=0.1 | mcs=25 ms=5  →  k=7  noise=37  sil=0.4785  score=0.4785
  [ 10/108] feat=base | nn=10 md=0.1 | mcs=30 ms=5  →  k=7  noise=37  sil=0.4785  score=0.4785
  [ 11/10

In [9]:
best = valid[
    (valid['features']          == 'base') &
    (valid['n_neighbors']       == 10)     &
    (valid['min_dist']          == 0.05)   &
    (valid['min_cluster_size']  == 30)     &
    (valid['min_samples']       == 5)
].iloc[0]

In [10]:
# cols = ['features', 'n_neighbors', 'min_dist', 'min_cluster_size',
#             'min_samples', 'n_clusters', 'n_noise', 'silhouette', 'score']
# print(valid[cols].head(10).to_string(index=False))

In [11]:
# best = valid.iloc[0]
print(f"\n{'='*72}")
print(f"BEST CONFIG:")
print(f"  features        = {best['features']}")
print(f"  n_neighbors     = {int(best['n_neighbors'])}")
print(f"  min_dist        = {best['min_dist']}")
print(f"  min_cluster_size= {int(best['min_cluster_size'])}")
print(f"  min_samples     = {int(best['min_samples'])}")
print(f"  → {int(best['n_clusters'])} clusters  |  "
        f"{int(best['n_noise'])} noise  |  "
        f"silhouette={best['silhouette']:.4f}")
print(f"{'='*72}")

print("\nPlotting best result...")
final_df = plot_best(df, best)


BEST CONFIG:
  features        = base
  n_neighbors     = 10
  min_dist        = 0.05
  min_cluster_size= 30
  min_samples     = 5
  → 6 clusters  |  17 noise  |  silhouette=0.5093

Plotting best result...
  Saved umap_best_result.png

── Cluster Summary (best config) ────────────────────────────────────────
 Cluster   Size     USG     AST    OREB    DREB      TS  Top players
-----------------------------------------------------------------------------------------------
   NOISE     17   0.151   0.160   0.054   0.125   0.543  Amen Thompson, Nic Claxton, Dyson Daniels, Gary Payton II
       0     77   0.243   0.243   0.021   0.099   0.573  Shai Gilgeous-Alexander, Donovan Mitchell, Cade Cunningham, Tyrese Maxey
       1     53   0.146   0.089   0.118   0.188   0.635  Jalen Duren, Robert Williams III, Chet Holmgren, Paul Reed
       2     61   0.243   0.191   0.051   0.180   0.585  Nikola Jokić, Giannis Antetokounmpo, Victor Wembanyama, Luka Dončić
       3    105   0.164   0.158   0.03

### Including player archetypes into training data

In [12]:
from src.enrich_lineups_with_clustering import load_data, assign_noise_players, build_archetype_lookup, enrich_lineups, build_compatibility_matrix, print_top_combos

In [13]:
players, lineups = load_data()
players = assign_noise_players(players)

Loading data...
  444 players  |  7920 lineups
  17 noise players

Assigning 17 noise players to nearest cluster via KNN...
  Amen Thompson                  → Athletic Role Player
  Ben Sheppard                   → 3-and-D Wing
  Caleb Martin                   → Spot-Up Shooter
  Craig Porter Jr.               → 3-and-D Wing
  Dyson Daniels                  → Star Versatile Engine
  Gary Payton II                 → Athletic Role Player
  Isaiah Livers                  → Athletic Role Player
  John Konchar                   → Athletic Role Player
  Jordan Clarkson                → Spot-Up Shooter
  Kelly Olynyk                   → Athletic Role Player
  Kelly Oubre Jr.                → Spot-Up Shooter
  Kevin McCullar Jr.             → 3-and-D Wing
  Nic Claxton                    → Athletic Role Player
  Nikola Jović                   → 3-and-D Wing
  Precious Achiuwa               → Athletic Role Player
  Sion James                     → Spot-Up Shooter
  Trendon Watford              

In [14]:
lookup           = build_archetype_lookup(players)
lineups_enriched = enrich_lineups(lineups, lookup)


Enriching lineups...
  7819/7920 lineups matched (98.7%)


In [15]:
lineups_enriched.to_csv('lineups_with_archetypes.csv', index=False)
print("  Saved lineups_with_archetypes.csv")

  Saved lineups_with_archetypes.csv


In [16]:
compatibility = build_compatibility_matrix(lineups_enriched)


── Archetype Compatibility Matrix (avg synergy_delta, 2-man) ──────────
                       Primary Ball-Handler  Rim-Runner  Star Versatile Engine  3-and-D Wing  Athletic Role Player  Spot-Up Shooter
Primary Ball-Handler                   0.41        0.33                   0.54         -0.46                 -0.01            -0.24
Rim-Runner                             0.33        0.04                   0.18         -0.51                 -0.31             0.83
Star Versatile Engine                  0.54        0.18                   0.50         -0.30                 -0.79             0.69
3-and-D Wing                          -0.46       -0.51                  -0.30         -0.88                 -0.51            -0.81
Athletic Role Player                  -0.01       -0.31                  -0.79         -0.51                  0.40            -0.75
Spot-Up Shooter                       -0.24        0.83                   0.69         -0.81                 -0.75            -0.48
  S

In [ ]:
print_top_combos(lineups_enriched, group_size=5)
print_top_combos(lineups_enriched, group_size=2)


Building compatibility matrix...

── Archetype Compatibility Matrix (avg synergy_delta, 2-man) ──────────
                       Primary Ball-Handler  Rim-Runner  Star Versatile Engine  3-and-D Wing  Athletic Role Player  Spot-Up Shooter
Primary Ball-Handler                   0.73        0.48                   0.73         -0.52                  0.03            -0.27
Rim-Runner                             0.48        1.45                   0.35         -0.63                 -0.42             1.29
Star Versatile Engine                  0.73        0.35                   0.98         -0.33                 -1.08             0.97
3-and-D Wing                          -0.52       -0.63                  -0.33         -1.15                 -0.62            -1.02
Athletic Role Player                   0.03       -0.42                  -1.08         -0.62                  0.77            -1.02
Spot-Up Shooter                       -0.27        1.29                   0.97         -1.02         

In [33]:
lineups_enriched.columns

Index(['GROUP_SET', 'GROUP_ID', 'GROUP_NAME', 'GP', 'W', 'L', 'W_PCT', 'MIN',
       'E_OFF_RATING', 'OFF_RATING', 'E_DEF_RATING', 'DEF_RATING',
       'E_NET_RATING', 'NET_RATING', 'AST_PCT', 'AST_TO', 'AST_RATIO',
       'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'TM_TOV_PCT', 'EFG_PCT', 'TS_PCT',
       'E_PACE', 'PACE', 'PACE_PER40', 'POSS', 'PIE', 'GP_RANK', 'W_RANK',
       'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'OFF_RATING_RANK',
       'DEF_RATING_RANK', 'NET_RATING_RANK', 'AST_PCT_RANK', 'AST_TO_RANK',
       'AST_RATIO_RANK', 'OREB_PCT_RANK', 'DREB_PCT_RANK', 'REB_PCT_RANK',
       'TM_TOV_PCT_RANK', 'EFG_PCT_RANK', 'TS_PCT_RANK', 'PACE_RANK',
       'PIE_RANK', 'SUM_TIME_PLAYED', 'season', 'group_size',
       'expected_net_rating', 'expected_off_rating', 'expected_def_rating',
       'expected_pie', 'expected_ts_usg_weighted', 'synergy_delta',
       'off_synergy_delta', 'def_synergy_delta', 'pie_synergy_delta',
       'player_ids_str', 'E_OFF_RATING_RANK', 'E_NET_RATING_RANK',
      